# AEGIS SC26 Figure Notebook

This notebook regenerates the SC26 submission figures directly from experiment artifacts.

It is designed for the reorganized repository layout and the current result schemas.

## Inputs

By default the notebook loads the newest `results/sc26_run_*` directory.

Optional environment overrides:
- `AEGIS_RESULTS_DIR`: explicit result bundle directory
- `AEGIS_PERF_ARTIFACT`: explicit path to `simulated_performance.json` when that appendix artifact is stored outside the main result bundle

## Outputs

The notebook writes the SC26 figure set into `figures/`:
- `baseline_comparison.png`
- `attack_results.png`
- `ablation_heatmap.png`
- `simulated_ablation_breakdown.png`
- `scaling_sweep.png` when simulated performance data is available
- `performance_overhead.png` when simulated performance data is available

Use the printed headline metrics below when updating the paper text and captions.


In [ ]:
import json
import os
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.ticker import FuncFormatter

plt.rcParams.update({
    'figure.dpi': 160,
    'savefig.dpi': 320,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.18,
    'grid.linewidth': 0.7,
    'axes.axisbelow': True,
    'legend.frameon': False,
})

PALETTE = {
    'blue': '#2f5d8a',
    'cyan': '#62a8ac',
    'green': '#4c956c',
    'amber': '#d9911f',
    'red': '#c8553d',
    'purple': '#7c6da8',
    'gray': '#6b7280',
    'light_blue': '#a7c7e7',
    'light_green': '#b8dfc2',
    'light_red': '#f1b6b6',
}

if Path.cwd().name == 'figures':
    ROOT = Path.cwd().resolve().parent
else:
    ROOT = Path.cwd().resolve()

FIG_DIR = ROOT / 'figures'
RESULTS_ROOT = ROOT / 'results'
RESULT_DIR_HINT = os.environ.get('AEGIS_RESULTS_DIR', '').strip()
PERF_ARTIFACT_HINT = os.environ.get('AEGIS_PERF_ARTIFACT', '').strip()

ATTACK_ORDER = ['filesystem', 'colocation', 'supply_chain', 'coordinated']
ATTACK_SHORT = ['FS Inj', 'CoLoc', 'Supply', 'CordEx']
ATTACK_LABELS = {
    'filesystem': 'Filesystem',
    'colocation': 'Co-Location',
    'supply_chain': 'Supply Chain',
    'coordinated': 'Coordinated',
}
ATTACK_COLORS = [PALETTE['blue'], PALETTE['green'], PALETTE['amber'], PALETTE['red']]
SIM_ATTACK_COLORS = [PALETTE['blue'], PALETTE['green'], PALETTE['amber'], PALETTE['red']]


def resolve_result_dir() -> Path:
    if RESULT_DIR_HINT:
        candidate = Path(RESULT_DIR_HINT)
        if not candidate.is_absolute():
            candidate = (ROOT / candidate).resolve()
        if candidate.exists():
            return candidate
        raise FileNotFoundError(f'AEGIS_RESULTS_DIR does not exist: {candidate}')
    candidates = sorted([p for p in RESULTS_ROOT.glob('sc26_run_*') if p.is_dir()], key=lambda p: p.name)
    if not candidates:
        raise FileNotFoundError('No results/sc26_run_* directories found under results/')
    return candidates[-1]


def resolve_perf_artifact(result_dir: Path):
    local = result_dir / 'simulated_performance.json'
    if local.exists():
        return local
    if PERF_ARTIFACT_HINT:
        candidate = Path(PERF_ARTIFACT_HINT)
        if not candidate.is_absolute():
            candidate = (ROOT / candidate).resolve()
        if candidate.exists():
            return candidate
        raise FileNotFoundError(f'AEGIS_PERF_ARTIFACT does not exist: {candidate}')
    return None


def save_fig(fig, filename: str) -> Path:
    path = FIG_DIR / filename
    fig.savefig(path, bbox_inches='tight')
    print(f'Saved {path.relative_to(ROOT)}')
    return path


def load_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(f'Missing artifact: {path}')
    return json.loads(path.read_text(encoding='utf-8'))


RESULT_DIR = resolve_result_dir()
PERF_PATH = resolve_perf_artifact(RESULT_DIR)
print(f'Using results directory: {RESULT_DIR.relative_to(ROOT)}')
if PERF_PATH is None:
    print('Simulated performance artifact: not found; scaling figures will be skipped.')
else:
    try:
        display_perf = PERF_PATH.relative_to(ROOT)
    except ValueError:
        display_perf = PERF_PATH
    print(f'Using simulated performance artifact: {display_perf}')


In [ ]:
def parse_baseline_table(path: Path):
    rows = []
    in_table = False
    for line in path.read_text(encoding='utf-8').splitlines():
        stripped = line.strip()
        if stripped.startswith('| Defense |'):
            in_table = True
            continue
        if in_table and stripped.startswith('|---------'):
            continue
        if in_table:
            if not stripped.startswith('|'):
                break
            cols = [c.strip() for c in stripped.strip('|').split('|')]
            if len(cols) != 7:
                continue
            rows.append({
                'defense': cols[0],
                'detections': [1 if 'DET' in c else 0 for c in cols[1:5]],
                'rate': float(cols[5].replace('%', '').strip()),
                'avg_time_ms': float(cols[6].split('±')[0].strip()),
            })
    if not rows:
        raise ValueError(f'Could not parse baseline comparison table: {path}')
    return rows


def collect_real_latency_summaries(result_dir: Path):
    items = []
    for path in sorted(result_dir.glob('real_latency_*.json')):
        if path.name == 'real_latency_sweep.json':
            continue
        payload = load_json(path)
        items.append(payload['summary'])
    items.sort(key=lambda item: ATTACK_ORDER.index(item['attack']))
    return items


def collect_microbenchmarks(result_dir: Path):
    items = []
    for path in sorted(result_dir.glob('bpf_microbenchmark_*.json')):
        payload = load_json(path)
        overhead = payload['overhead']
        throughput_loss = overhead.get('throughput_loss_pct', abs(overhead['ops_per_sec_pct']))
        items.append({
            'mode': payload['config']['mode'],
            'elapsed_pct': overhead['elapsed_pct'],
            'task_clock_pct': overhead['task_clock_pct'],
            'throughput_delta_pct': overhead.get('throughput_delta_pct', overhead['ops_per_sec_pct']),
            'throughput_loss_pct': throughput_loss,
        })
    mode_order = {'openat': 0, 'read': 1, 'connect': 2, 'execve': 3}
    items.sort(key=lambda row: mode_order.get(row['mode'], 99))
    return items


def load_simulated_ablation_matrix(payload):
    config_names = list(payload['results'].keys())
    attack_names = list(next(iter(payload['results'].values()))['results'].keys())
    matrix = np.zeros((len(config_names), len(attack_names)))
    rates = []
    for i, cfg in enumerate(config_names):
        rates.append(payload['results'][cfg]['detection_rate'])
        for j, attack in enumerate(attack_names):
            matrix[i, j] = 100.0 if payload['results'][cfg]['results'][attack]['detected'] else 0.0
    return config_names, attack_names, matrix, rates


baseline_rows = parse_baseline_table(RESULT_DIR / 'baseline_comparison.md')
real_latency = collect_real_latency_summaries(RESULT_DIR)
real_latency_sweep = load_json(RESULT_DIR / 'real_latency_sweep.json')
real_ablation = load_json(RESULT_DIR / 'real_ablation.json')
simulated_all = load_json(RESULT_DIR / 'simulated_all_attacks.json')
simulated_ablation = load_json(RESULT_DIR / 'simulated_ablation.json')
simulated_fp = load_json(RESULT_DIR / 'simulated_false_positive.json')
simulated_perf = load_json(PERF_PATH) if PERF_PATH else None
microbench = collect_microbenchmarks(RESULT_DIR)

print('Artifacts loaded:')
print(f'  baseline comparison rows: {len(baseline_rows)}')
print(f'  real attack summaries:    {len(real_latency)}')
print(f'  microbenchmark modes:     {len(microbench)}')
print(f'  simulated attacks:        {len(simulated_all["results"])}')
print(f'  benign workflows:         {simulated_fp["workflow_count"]}')
print(f'  scaling artifact:         {"present" if simulated_perf else "missing"}')


In [ ]:
# Headline metrics for the paper text and figure captions.

real_latency_range = [row['median_detection_latency_ms'] for row in real_latency]
real_cpu_range = [row['median_cpu_overhead_percent'] for row in real_latency]
real_exfil_values = [row['median_exfiltrated_bytes'] for row in real_latency]

best_baseline = max((row for row in baseline_rows if row['defense'] != 'AEGIS (Ours)'), key=lambda row: row['rate'])
ours_baseline = next(row for row in baseline_rows if row['defense'] == 'AEGIS (Ours)')

print('SC26 headline metrics')
print('=====================')
print(f"Real-path attacks detected: {sum(row['detected_trials'] == row['repeats'] for row in real_latency)}/{len(real_latency)}")
print(f"Real-path latency range: {min(real_latency_range):.2f} ms to {max(real_latency_range):.2f} ms")
print(f"Real-path exfiltration range: {min(real_exfil_values):.0f} B to {max(real_exfil_values):.0f} B")
print(f"Real-path CPU overhead max: {max(real_cpu_range):.4f}%")
print()
for row in real_latency:
    print(f"  {ATTACK_LABELS[row['attack']]:<12} latency={row['median_detection_latency_ms']:.2f} ms, exfil={row['median_exfiltrated_bytes']:.0f} B, cpu={row['median_cpu_overhead_percent']:.4f}%")
print()
print(f"Simulated coverage: {simulated_all['aggregate']['attacks_detected']}/{simulated_all['aggregate']['experiments_completed']} attacks detected, {simulated_all['aggregate']['detection_rate_percent']:.0f}% detection rate")
print(f"Simulated detections: {simulated_all['aggregate']['total_detections']} total findings, {simulated_all['aggregate']['total_exfiltrated_bytes']} B exfiltrated")
print(f"False positives: {simulated_fp['total_false_positives']} across {simulated_fp['total_actions']} benign actions in {simulated_fp['workflow_count']} workflows")
print()
print(f"Baseline comparison: AEGIS rate={ours_baseline['rate']:.0f}% at {ours_baseline['avg_time_ms']:.4f} ms; strongest baseline rate={best_baseline['rate']:.0f}% ({best_baseline['defense']})")
print('Microbenchmark elapsed overheads:')
for row in microbench:
    print(f"  {row['mode']:<8} elapsed={row['elapsed_pct']:.2f}% task-clock={row['task_clock_pct']:.2f}% throughput loss={row['throughput_loss_pct']:.2f}%")
if simulated_perf:
    mixed_1s_10 = next(row for row in simulated_perf['agent_count_sweep'] if row['agent_count'] == 10)
    print()
    print(f"Scaling appendix: {simulated_perf['low_overhead_count']}/{simulated_perf['total_configurations']} configurations remain below 5% overhead")
    print(f"Mixed workload at 1.0 s, 10 agents: {mixed_1s_10['overhead_percent']:.2f}% overhead")


In [ ]:
# Figure 1: baseline comparison.

ordered_baseline_rows = sorted(baseline_rows, key=lambda row: 0 if row['defense'] == 'AEGIS (Ours)' else 1)
defenses = [row['defense'] for row in ordered_baseline_rows]
mat = np.array([row['detections'] for row in ordered_baseline_rows], dtype=float)
rates = np.array([row['rate'] for row in ordered_baseline_rows], dtype=float)
analysis_times = np.array([row['avg_time_ms'] for row in ordered_baseline_rows], dtype=float)
colors = [PALETTE['blue'] if d != 'AEGIS (Ours)' else PALETTE['amber'] for d in defenses]

fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(14.2, 5.2), gridspec_kw={'width_ratios': [4.6, 1.5, 1.9]}, constrained_layout=True)
heatmap_cmap = ListedColormap(['#d9d9d9', PALETTE['green']])
ax0.imshow(mat, cmap=heatmap_cmap, vmin=0, vmax=1, aspect='auto')
ax0.set_xticks(range(len(ATTACK_SHORT)))
ax0.set_xticklabels(['Filesystem\nInjection', 'Co-Location\nInjection', 'Supply Chain\nInjection', 'Coordinated\nExfiltration'])
for tick in ax0.get_xticklabels():
    tick.set_color('#222222')
    tick.set_fontweight('normal')
ax0.set_yticks(range(len(defenses)))
ax0.set_yticklabels(defenses)
ax0.set_title('Detection Outcomes by Defense')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax0.text(j, i, 'DET' if mat[i, j] else 'MISS', ha='center', va='center', fontsize=10, fontweight='bold')
for tick in ax0.get_yticklabels():
    if tick.get_text() == 'AEGIS (Ours)':
        tick.set_fontweight('bold')

ypos = np.arange(len(defenses))
rate_bars = ax1.barh(ypos, rates, color=colors)
ax1.set_title('Detection Rate')
ax1.set_xlim(0, 100)
ax1.set_xlabel('Percent')
ax1.set_yticks(ypos)
ax1.set_yticklabels([])
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.2)
for bar, rate in zip(rate_bars, rates):
    x = min(rate + 2.0, 98.0) if rate > 0 else 2.0
    ax1.text(x, bar.get_y() + bar.get_height() / 2, f'{rate:.0f}%', va='center', ha='left', fontsize=9, fontweight='bold')

time_bars = ax2.barh(ypos, analysis_times, color=colors)
ax2.set_title('Mean Analysis Time')
ax2.set_xlabel('Milliseconds')
ax2.set_yticks(ypos)
ax2.set_yticklabels([])
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.2)
for bar, value in zip(time_bars, analysis_times):
    ax2.text(bar.get_width() + max(analysis_times) * 0.02, bar.get_y() + bar.get_height() / 2, f'{value:.4f}', va='center', ha='left', fontsize=9, fontweight='bold')

save_fig(fig, 'baseline_comparison.png')
plt.show()


In [ ]:
# Figure 2: measured per-attack outcomes on the real path.

attack_names = [ATTACK_LABELS[item['attack']] for item in real_latency]
latencies = [item['median_detection_latency_ms'] for item in real_latency]
exfil_bytes = [item['median_exfiltrated_bytes'] for item in real_latency]
cpu_overhead = [item['median_cpu_overhead_percent'] for item in real_latency]
detected_text = [f"{item['detected_trials']}/{item['repeats']}" for item in real_latency]
colors = [PALETTE['blue'], PALETTE['green'], PALETTE['amber'], PALETTE['red']]

fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.7), constrained_layout=True)

bars0 = axes[0].bar(attack_names, latencies, color=colors)
axes[0].set_title('Measured Detection Latency')
axes[0].set_ylabel('Milliseconds')
axes[0].tick_params(axis='x', rotation=18)
for bar, value in zip(bars0, latencies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 6, f'{value:.0f}', ha='center', va='bottom', fontsize=9)

bars1 = axes[1].bar(attack_names, exfil_bytes, color=colors)
axes[1].set_title('Exfiltration Before Detection')
axes[1].set_ylabel('Bytes')
axes[1].tick_params(axis='x', rotation=18)
for bar, value in zip(bars1, exfil_bytes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 12, f'{value:.0f}', ha='center', va='bottom', fontsize=9)

bars2 = axes[2].bar(attack_names, cpu_overhead, color=colors)
axes[2].set_title('Verifier CPU Overhead')
axes[2].set_ylabel('Percent')
axes[2].tick_params(axis='x', rotation=18)
axes[2].yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x:.02f}%'))
for bar, text in zip(bars2, detected_text):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0012, text, ha='center', va='bottom', fontsize=9)

save_fig(fig, 'attack_results.png')
plt.show()


In [ ]:
# Figure 3: real ablation matrix with a separate exfiltration panel.

from matplotlib.colors import ListedColormap

config_keys = real_ablation['configs']
attack_keys = real_ablation['attacks']
summary = real_ablation['summary']
lookup = {(row['config_key'], row['attack_key']): row for row in summary}
config_names = [next(row['config_name'] for row in summary if row['config_key'] == key) for key in config_keys]
display_attack_names = [
    'Filesystem\nInjection',
    'Co-Location\nInjection',
    'Supply Chain\nInjection',
    'Coordinated\nExfiltration',
]
status_matrix = np.zeros((len(config_keys), len(attack_keys)), dtype=int)
exfil_matrix = np.full((len(config_keys), len(attack_keys)), np.nan)
status_annotations = [['' for _ in attack_keys] for _ in config_keys]
exfil_annotations = [['' for _ in attack_keys] for _ in config_keys]

for i, cfg_key in enumerate(config_keys):
    for j, attack_key in enumerate(attack_keys):
        row = lookup[(cfg_key, attack_key)]
        detected = int(row['detected_trials'] == row['total_trials'])
        status_matrix[i, j] = detected
        exfil = row['median_exfiltrated_bytes']
        status_annotations[i][j] = 'DET' if detected else 'MISS'
        if exfil is not None:
            exfil_matrix[i, j] = exfil
            exfil_annotations[i][j] = f'{int(exfil)} B'
        else:
            exfil_annotations[i][j] = 'N/A'

fig = plt.figure(figsize=(14.4, 5.4), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.04])
ax_status = fig.add_subplot(gs[0, 0])
ax_exfil = fig.add_subplot(gs[0, 1])
cax = fig.add_subplot(gs[0, 2])

status_cmap = ListedColormap(['#d9d9d9', PALETTE['green']])
ax_status.imshow(status_matrix, cmap=status_cmap, vmin=0, vmax=1, aspect='auto')
ax_status.set_title('Detection Outcome')
ax_status.set_xticks(range(len(attack_keys)))
ax_status.set_xticklabels(display_attack_names)
for tick in ax_status.get_xticklabels():
    tick.set_color('#222222')
    tick.set_fontweight('normal')
ax_status.set_yticks(range(len(config_names)))
ax_status.set_yticklabels(['AEGIS(Ours)' if name == 'Full AEGIS' else name for name in config_names])
for tick in ax_status.get_yticklabels():
    if tick.get_text() == 'AEGIS(Ours)':
        tick.set_fontweight('bold')
ax_status.tick_params(axis='x', length=0, pad=10)
ax_status.tick_params(axis='y', length=0)
ax_status.set_xticks(np.arange(-0.5, len(attack_keys), 1), minor=True)
ax_status.set_yticks(np.arange(-0.5, len(config_names), 1), minor=True)
ax_status.grid(which='minor', color='white', linestyle='-', linewidth=2)
ax_status.tick_params(which='minor', bottom=False, left=False)
for i in range(len(config_names)):
    for j in range(len(attack_keys)):
        ax_status.text(j, i, status_annotations[i][j], ha='center', va='center', fontsize=10, fontweight='bold', color='#111111')
exfil_cmap = plt.cm.YlOrBr.copy()
exfil_cmap.set_bad('#ececec')
im_exfil = ax_exfil.imshow(exfil_matrix, cmap=exfil_cmap, aspect='auto')
ax_exfil.set_title('Median Exfiltration Before Containment')
ax_exfil.set_xticks(range(len(attack_keys)))
ax_exfil.set_xticklabels(display_attack_names)
for tick in ax_exfil.get_xticklabels():
    tick.set_color('#222222')
    tick.set_fontweight('normal')
ax_exfil.set_yticks(range(len(config_names)))
ax_exfil.set_yticklabels([])
ax_exfil.tick_params(axis='x', length=0, pad=10)
ax_exfil.tick_params(axis='y', length=0)
ax_exfil.set_xticks(np.arange(-0.5, len(attack_keys), 1), minor=True)
ax_exfil.set_yticks(np.arange(-0.5, len(config_names), 1), minor=True)
ax_exfil.grid(which='minor', color='white', linestyle='-', linewidth=2)
ax_exfil.tick_params(which='minor', bottom=False, left=False)
for i in range(len(config_names)):
    for j in range(len(attack_keys)):
        cell_value = exfil_matrix[i, j]
        text_color = '#222222' if np.isnan(cell_value) or cell_value < 220 else 'white'
        ax_exfil.text(j, i, exfil_annotations[i][j], ha='center', va='center', fontsize=10, fontweight='bold', color=text_color)
cbar = fig.colorbar(im_exfil, cax=cax)
cbar.set_label('Bytes')

save_fig(fig, 'ablation_heatmap.png')
plt.show()


In [ ]:
# Figure 6: simulated ablation breakdown.

from matplotlib.colors import ListedColormap

config_names, attack_names, sim_ablation_matrix, sim_rates = load_simulated_ablation_matrix(simulated_ablation)
display_attack_names = [
    'Volume\nAttack',
    'Sensitive\nFile',
    'Covert\nChannel',
    'Injection\nAttack',
]
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13.4, 5.3), gridspec_kw={'width_ratios': [1.65, 1.0]}, constrained_layout=True)

binary_cmap = ListedColormap(['#d9d9d9', PALETTE['green']])
ax0.imshow(sim_ablation_matrix, cmap=binary_cmap, vmin=0, vmax=100, aspect='auto')
ax0.set_xticks(range(len(display_attack_names)))
ax0.set_xticklabels(display_attack_names)
for tick in ax0.get_xticklabels():
    tick.set_color('#222222')
    tick.set_fontweight('normal')
ax0.set_yticks(range(len(config_names)))
ax0.set_yticklabels(['AEGIS(Ours)' if name == 'Full AEGIS' else name for name in config_names])
for tick in ax0.get_yticklabels():
    if tick.get_text() == 'AEGIS(Ours)':
        tick.set_fontweight('bold')
ax0.set_title('Per-Attack Detection Outcome')
ax0.tick_params(axis='x', length=0, pad=10)
ax0.tick_params(axis='y', length=0)
ax0.set_xticks(np.arange(-0.5, len(display_attack_names), 1), minor=True)
ax0.set_yticks(np.arange(-0.5, len(config_names), 1), minor=True)
ax0.grid(which='minor', color='white', linestyle='-', linewidth=2)
ax0.tick_params(which='minor', bottom=False, left=False)
for i in range(len(config_names)):
    for j in range(len(display_attack_names)):
        detected = sim_ablation_matrix[i, j] == 100
        ax0.text(j, i, 'DET' if detected else 'MISS', ha='center', va='center', fontsize=9.5, fontweight='bold', color='#111111')

bar_colors = []
for name in config_names:
    if name == 'Full AEGIS':
        bar_colors.append(PALETTE['amber'])
    elif name == 'Minimal (path only)':
        bar_colors.append('#b8c2cc')
    else:
        bar_colors.append(PALETTE['blue'])
ypos = np.arange(len(config_names))
bars = ax1.barh(ypos, sim_rates, color=bar_colors)
ax1.set_title('Overall Detection Rate')
ax1.set_yticks(ypos)
ax1.set_yticklabels([])
ax1.set_xlabel('Percent (%)')
ax1.set_xlim(0, 112)
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.18)
for bar, rate in zip(bars, sim_rates):
    y = bar.get_y() + bar.get_height() / 2
    ax1.text(min(rate + 1.5, 101.0), y, f'{rate:.0f}%', va='center', ha='left', fontsize=10, fontweight='bold')

save_fig(fig, 'simulated_ablation_breakdown.png')
plt.show()


In [ ]:
# Figure 7: scaling and workload sweeps from the simulated performance study.

if simulated_perf is None:
    print('Skipping scaling_sweep.png because simulated_performance.json is unavailable.')
else:
    interval_sweep = sorted(simulated_perf['interval_sweep'], key=lambda row: row['attestation_interval'])
    agent_sweep = sorted(simulated_perf['agent_count_sweep'], key=lambda row: row['agent_count'])
    workload_sweep = simulated_perf['workload_type_sweep']

    fig, axes = plt.subplots(1, 3, figsize=(14.6, 4.8), constrained_layout=True)

    axes[0].plot([row['attestation_interval'] for row in interval_sweep], [row['overhead_percent'] for row in interval_sweep], marker='o', linewidth=2.2, color=PALETTE['blue'])
    axes[0].set_xscale('log')
    axes[0].set_title('Overhead vs Interval')
    axes[0].set_xlabel('Attestation interval (s)')
    axes[0].set_ylabel('Overhead (%)')

    axes[1].plot([row['agent_count'] for row in agent_sweep], [row['overhead_percent'] for row in agent_sweep], marker='o', linewidth=2.2, color=PALETTE['green'])
    axes[1].set_title('Overhead vs Agent Count')
    axes[1].set_xlabel('Agent count')
    axes[1].set_ylabel('Overhead (%)')

    workload_colors = [PALETTE['blue'], PALETTE['green'], PALETTE['amber'], PALETTE['red']][:len(workload_sweep)]
    workload_labels = [row['workload_type'] for row in workload_sweep]
    workload_values = [row['overhead_percent'] for row in workload_sweep]
    bars = axes[2].bar(workload_labels, workload_values, color=workload_colors)
    axes[2].set_title('Overhead by Workload')
    axes[2].set_ylabel('Overhead (%)')
    axes[2].tick_params(axis='x', rotation=18)
    for tick in axes[2].get_xticklabels():
        tick.set_color('#222222')
        tick.set_fontweight('normal')
    for bar, value in zip(bars, workload_values):
        axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.08, f'{value:.2f}%', ha='center', va='bottom', fontsize=9, fontweight='bold', color='black')

    save_fig(fig, 'scaling_sweep.png')
    plt.show()


In [ ]:
# Figure 8: combined measured-overhead summary used in the paper draft.

sweep_summary = sorted(real_latency_sweep['summary'], key=lambda row: row['interval'])
intervals = [row['interval'] for row in sweep_summary]
avg_latency = [row['avg_latency_ms'] for row in sweep_summary]
cpu_pct = [row['cpu_overhead'] for row in sweep_summary]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13.4, 5.1), constrained_layout=True)

mode_names = [row['mode'] for row in microbench]
elapsed_pct = [row['elapsed_pct'] for row in microbench]
task_clock_pct = [row['task_clock_pct'] for row in microbench]
throughput_loss = [row['throughput_loss_pct'] for row in microbench]
bar_x = np.arange(len(mode_names))
width = 0.24

ax0.bar(bar_x - width, elapsed_pct, width=width, label='Elapsed overhead', color=PALETTE['blue'])
ax0.bar(bar_x, task_clock_pct, width=width, label='Task-clock overhead', color=PALETTE['green'])
ax0.bar(bar_x + width, throughput_loss, width=width, label='Throughput loss', color=PALETTE['amber'])
ax0.set_xticks(bar_x)
ax0.set_xticklabels(mode_names)
ax0.set_ylabel('Percent (%)')
ax0.set_title('Direct Probe Overhead by Syscall Family')
ax0.grid(axis='y', alpha=0.18)
ax0.legend(frameon=False, ncol=1, loc='upper right')
for xpos, value in zip(bar_x - width, elapsed_pct):
    ax0.text(xpos, value + 0.7, f'{value:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold', color='black')
for xpos, value in zip(bar_x, task_clock_pct):
    ax0.text(xpos, value + 0.7, f'{value:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold', color='black')
for xpos, value in zip(bar_x + width, throughput_loss):
    ax0.text(xpos, value + 0.7, f'{value:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold', color='black')

ax1.plot(intervals, avg_latency, marker='o', markersize=7, linewidth=2.4, color=PALETTE['red'], label='Detection latency (ms)')
ax1.set_xscale('log')
ax1.set_xlabel('Attestation interval (s)')
ax1.set_ylabel('Detection latency (ms)', color=PALETTE['red'])
ax1.tick_params(axis='y', labelcolor=PALETTE['red'])
ax1.set_title('Latency and CPU vs Attestation Interval')
ax1.grid(axis='both', alpha=0.16)
ax1b = ax1.twinx()
ax1b.plot(intervals, cpu_pct, marker='s', markersize=6, linestyle='--', linewidth=2.1, color=PALETTE['green'], label='CPU overhead (%)')
ax1b.set_ylabel('CPU overhead (%)', color=PALETTE['green'])
ax1b.tick_params(axis='y', labelcolor=PALETTE['green'])

for idx in [0, 2, len(intervals) - 1]:
    ax1.annotate(f'{avg_latency[idx]:.0f} ms', (intervals[idx], avg_latency[idx]), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9, color=PALETTE['red'], fontweight='bold')

cpu_offsets = {0: (0, -16), 2: (0, -22), len(intervals) - 1: (0, -18)}
for idx, (dx, dy) in cpu_offsets.items():
    ax1b.annotate(
        f'{cpu_pct[idx]:.3f}%',
        (intervals[idx], cpu_pct[idx]),
        textcoords='offset points',
        xytext=(dx, dy),
        ha='center',
        fontsize=9,
        color=PALETTE['green'],
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.85),
    )

lines = ax1.get_lines() + ax1b.get_lines()
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc='upper center', bbox_to_anchor=(0.5, 0.98), frameon=False)

save_fig(fig, 'performance_overhead.png')
plt.show()


In [ ]:
expected = [
    'baseline_comparison.png',
    'attack_results.png',
    'ablation_heatmap.png',
    'simulated_ablation_breakdown.png',
]
expected.append('performance_overhead.png')
if simulated_perf is not None:
    expected.append('scaling_sweep.png')

print('Generated figure files:')
for name in expected:
    path = FIG_DIR / name
    if path.exists():
        print(f'  {name}: {path.stat().st_size / 1024:.1f} KB')
    else:
        print(f'  {name}: missing')
